# Dino RoPE Frame-Index Debug

This notebook verifies the recent changes for:
- multi-image input with `load_indices=[0, -1]`
- frame-index-aware temporal RoPE in cross-attention
- end-to-end runability for 10 train steps


In [1]:
import os
from itertools import cycle

import torch
from torch.utils.data import DataLoader

from hydra import compose, initialize_config_module
from hydra.utils import instantiate

from hmr4d.configs import register_store_gvhmr
from hmr4d.dataset.imgfeat_motion.uni3c_aligned import Uni3CAlignedDatasetV1
from hmr4d.datamodule.mocap_trainX_testY import collate_fn

os.environ.setdefault('CUDA_VISIBLE_DEVICES', '3')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device =', device)
torch.set_float32_matmul_precision('high')


device = cuda


/home/guangyu/anaconda3/envs/hmr/lib/python3.10/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [2]:
with initialize_config_module(version_base='1.3', config_module='hmr4d.configs'):
    register_store_gvhmr()
    cfg = compose(config_name='train', overrides=['exp=gvhmr/test/test_dinov3'])

# keep this debug run lightweight
cfg.pipeline.args.dinov3_pool = 'none'   # expect (B, T, C, H, W), e.g. H=W=32
cfg.pipeline.args.dinov3_use_amp = False

model = instantiate(cfg.model, _recursive_=False).to(device)
model.train()
print('denoiser =', type(model.pipeline.denoiser3d).__name__)
print('dinov3_pool =', model.pipeline.dinov3_pool)


/home/guangyu/patrick/GVHMR/hmr4d/model/gvhmr/utils/postprocess.py:19: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @autocast(enabled=False)
/home/guangyu/patrick/GVHMR/hmr4d/model/gvhmr/utils/postprocess.py:61: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @autocast(enabled=False)
/home/guangyu/patrick/GVHMR/hmr4d/model/gvhmr/utils/postprocess.py:121: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @autocast(enabled=False)
/home/guangyu/patrick/GVHMR/hmr4d/network/base_arch/embeddings/rotary_embedding_v2.py:118: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @autocast(enabled=False)
/home/guangyu/patrick/GVHMR/hmr4d/model/gvhmr/pipeline/gvhmr_pipeline_dinov3.py:532: FutureWarn

denoiser = NetworkEncoderRoPEwithCA
dinov3_pool = none


In [3]:
dataset = Uni3CAlignedDatasetV1(
    random=True,
    n_els=256,
    load_image=True,
    load_indices=[0, -1],
)

loader = DataLoader(
    dataset,
    batch_size=4,
    shuffle=True,
    num_workers=0,
    drop_last=True,
    collate_fn=collate_fn,
)

def to_device(x, device):
    if isinstance(x, torch.Tensor):
        return x.to(device, non_blocking=True)
    if isinstance(x, dict):
        return {k: to_device(v, device) for k, v in x.items()}
    if isinstance(x, list):
        return [to_device(v, device) for v in x]
    if isinstance(x, tuple):
        return tuple(to_device(v, device) for v in x)
    return x

batch0 = next(iter(loader))
print('image shape:', None if batch0['image'] is None else tuple(batch0['image'].shape))
print('image_frame_indices shape:', tuple(batch0['image_frame_indices'].shape))
print('image_frame_indices sample:', batch0['image_frame_indices'][0].tolist())


[02/13 16:43:10][INFO] [UNI3C_SynthGenerated] Loading from inputs/uni3c_aligned ...
[02/13 16:43:10][INFO] [UNI3C_SynthGenerated] Loaded from 342 synthetic samples
[02/13 16:43:10][INFO] [UNI3C_SynthGenerated] Using 256 synthetic samples


image shape: (4, 2, 512, 512, 3)
image_frame_indices shape: (4, 2)
image_frame_indices sample: [0, 119]


/home/guangyu/patrick/GVHMR/hmr4d/dataset/imgfeat_motion/uni3c_aligned.py:82: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  batch = torch.load(self.root / f"{mid}/batch_meta

In [ ]:
batch_probe['K_fullimg'].shape

In [ ]:
dino_feat.shape

In [ ]:
K_crop.shape

In [ ]:
tmp = batch_probe['image'].permute(0, 1, 4, 2, 3).contiguous()
tmp = tmp.view(4 * 2, 3, tmp.shape[-2], tmp.shape[-1]).float()
tmp = ((tmp / 255.) - model.pipeline.dino_image_mean) / model.pipeline.dino_image_std

with torch.no_grad():
    dino_feat = model.pipeline.dinov3({"img": tmp})  # (B*L, C, H', W')
    
K_crop = model.pipeline._compute_crop_intrinsics(
    batch_probe["K_fullimg"].to(dino_feat.device),
    batch_probe["bbx_xys"].to(dino_feat.device),
    model.pipeline.dinov3_crop_size,
)

In [4]:
# Probe Dino feature tensor before full training loop
with torch.no_grad():
    batch_probe = to_device(batch0, device)
    dino_feat, _ = model.pipeline._compute_dinov3_embedding(batch_probe)

print('dino_feat shape:', None if dino_feat is None else tuple(dino_feat.shape))
if dino_feat is not None and dino_feat.dim() == 5:
    B, T, C, H, W = dino_feat.shape
    print('B,T,C,H,W =', (B, T, C, H, W))
    print('expected context token count (T*H*W):', T * H * W)


/home/guangyu/patrick/GVHMR/hmr4d/model/gvhmr/pipeline/gvhmr_pipeline_dinov3.py:220: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=self.dinov3_use_amp):


dino_feat shape: (4, 2, 1280, 32, 32)
B,T,C,H,W = (4, 2, 1280, 32, 32)
expected context token count (T*H*W): 2048


In [5]:
# 10 training steps: runability + latent/output dimensions
optim = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-5, weight_decay=1e-4)

it = cycle(loader)
for step in range(10):
    batch = to_device(next(it), device)

    assert batch['image_frame_indices'].shape[1] == 2, 'Expected load_indices=[0,-1] -> 2 frames'

    optim.zero_grad(set_to_none=True)
    outputs = model.training_step(batch, step)
    loss = outputs['loss']

    if not torch.isfinite(loss):
        raise RuntimeError(f'Non-finite loss at step {step}: {loss.item()}')

    loss.backward()
    optim.step()

    pred_ctx = outputs['model_output']['pred_context']
    pred_x = outputs['model_output']['pred_x']
    frame_idx0 = batch['image_frame_indices'][0].detach().cpu().tolist()

    print(
        f'step={step:02d} '
        f'loss={loss.item():.4f} '
        f'pred_context={tuple(pred_ctx.shape)} '
        f'pred_x={tuple(pred_x.shape)} '
        f'frame_idx[0]={frame_idx0}'
    )

print('Finished 10 training steps successfully.')


/home/guangyu/anaconda3/envs/hmr/lib/python3.10/site-packages/pytorch_lightning/core/module.py:451: You are trying to `self.log()` but the `self.trainer` reference is not registered on the model yet. This is most likely because the model hasn't been passed to the `Trainer`


step=00 loss=80.3708 pred_context=(4, 120, 512) pred_x=(4, 120, 151) frame_idx[0]=[0, 119]
step=01 loss=66.4899 pred_context=(4, 120, 512) pred_x=(4, 120, 151) frame_idx[0]=[0, 119]
step=02 loss=66.9053 pred_context=(4, 120, 512) pred_x=(4, 120, 151) frame_idx[0]=[0, 119]
step=03 loss=66.3799 pred_context=(4, 120, 512) pred_x=(4, 120, 151) frame_idx[0]=[0, 119]
step=04 loss=41.1190 pred_context=(4, 120, 512) pred_x=(4, 120, 151) frame_idx[0]=[0, 119]
step=05 loss=51.3915 pred_context=(4, 120, 512) pred_x=(4, 120, 151) frame_idx[0]=[0, 119]
step=06 loss=41.7122 pred_context=(4, 120, 512) pred_x=(4, 120, 151) frame_idx[0]=[0, 119]
step=07 loss=47.2059 pred_context=(4, 120, 512) pred_x=(4, 120, 151) frame_idx[0]=[0, 119]
step=08 loss=63.2771 pred_context=(4, 120, 512) pred_x=(4, 120, 151) frame_idx[0]=[0, 119]
step=09 loss=48.1577 pred_context=(4, 120, 512) pred_x=(4, 120, 151) frame_idx[0]=[0, 119]
Finished 10 training steps successfully.
